# Active cCREs vs open/closed chromatin (Fisher's exact)

Fisher's exact test of the contingency table of MPRA-active vs non-active cCREs against overlap with chondrocyte open chromatin. Active cCREs are ~3-fold more likely to lie in open than closed chromatin (P < 1e-308).

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Paths (EDIT): inputs read / outputs written by this notebook ---
ANNOTATIONS_CSV = '/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv'  # hMPRA per-oligo annotations
output_path = '.'  # where plots/tables are written

# Fisher's exact for active oligos × chondrocyte open chromatin

In [ ]:
# Read the hMPRA per-oligo annotations
oligos = pd.read_csv(ANNOTATIONS_CSV, header=0)
print('Number of oligos in hMPRA:', len(oligos))
oligos = oligos.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
print('Number of oligos in hMPRA without duplicates:', len(oligos))

min_DNA_counts = 50

oligos = oligos[(oligos['DNA_counts_raw_derived']>min_DNA_counts) &
                               (oligos['DNA_counts_raw_ancestral']>min_DNA_counts)]
print('Number of oligos in hMPRA after filtering for at least 50 DNA counts in both archaic and derived:', len(oligos))

In [ ]:
from scipy.stats import fisher_exact

# Step 1: Define activity status (NA = not active; otherwise active)
oligos['activity_status'] = oligos['differential_activity'].apply(
    lambda x: 'Not Active' if pd.isna(x) else 'Active'
)

# Step 2: Define chromatin state (open vs closed)
oligos['chromatin_state'] = oligos.apply(
    lambda row: 'Closed' if (
        row['ATACseq_peaks_fetal_chondrocytes'] != 0 and
        row['H3K27Ac_peaks_fetal_chondrocytes'] != 0
    ) else 'Open',
    axis=1
)

# Step 3: Create contingency table
contingency_table = pd.crosstab(
    oligos['activity_status'], 
    oligos['chromatin_state'],
    margins=True
)

print("Contingency Table: Active/Non-Active Oligos X Open/Close Chromatin")
print("="*60)
print(contingency_table)
print("\n")

# Step 4: Extract 2x2 table for Fisher's exact test
table_2x2 = pd.crosstab(oligos['activity_status'], oligos['chromatin_state'])
print("2x2 Contingency Table (without margins):")
print(table_2x2)
print("\n")

# Step 5: Apply Fisher's exact test
oddsratio, p_value = fisher_exact(table_2x2)

print("Fisher's Exact Test Results:")
print("="*60)
print(f"Odds Ratio: {oddsratio:.4f}")
print(f"P-value: {p_value:.4e}")
print(f"Significant at α=0.05: {'Yes' if p_value < 0.05 else 'No'}")

In [ ]:
# Get the actual raw p_value
print(f"Exact P-value with full precision: {p_value}")
print(f"P-value represented in bits: {p_value.hex() if hasattr(p_value, 'hex') else 'N/A'}")
print(f"\nSmallest positive float Python can represent: {sys.float_info.min}")

# Log p-value for better precision handling
log_pval = np.log10(p_value) if p_value > 0 else "P-value is exactly 0 or below machine precision"
print(f"\nLog10(P-value): {log_pval}")

# The p-value underflows to 0.0 -> smaller than the smallest positive float64
print(f"\nConclusion: The P-value is smaller than the smallest positive float")
print(f"that can be represented (< {sys.float_info.min})")
print(f"Report: P-value < 1e-308")